# L5 · Lab B — Token-Level Rewards & Advantages

**RL for LLMs / RLHF — Vizuara AI Labs · Dr. Rajat Dandekar**

This is the **code version of the worked example** from the lecture (slides 50–52). PPO updates *every token* by its own advantage, so we must turn one end-of-sequence reward-model score into a **per-token** signal. You will compute, on a real generated sentence:

`R_t = −β·log(π_θ/π_ref)` per token, plus the reward-model score on the **last** token → reward-to-go `G_t` → advantage `A_t = G_t − baseline`.

> Runtime: **GPU**. ~3 minutes.

In [1]:
# !pip -q install "transformers==4.45.2" "datasets==3.0.1" >/dev/null

In [6]:
# ✅ PROVIDED — get an HH reward model (load Lab A's ./rm, else quick-train)
import os, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from datasets import load_dataset
device = "cuda" if torch.cuda.is_available() else "cpu"
rm_tok = AutoTokenizer.from_pretrained("gpt2"); rm_tok.pad_token = rm_tok.eos_token; rm_tok.truncation_side = "left"
if os.path.isdir("rm"):
    rm = AutoModelForSequenceClassification.from_pretrained("rm").to(device).eval()
else:
    rm = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=1)
    rm.config.pad_token_id = rm_tok.eos_token_id; rm.to(device)
    ds = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=0).select(range(4000))
    def _sc(t):
        e = rm_tok(t, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        return rm(**e).logits.squeeze(-1)
    opt = torch.optim.AdamW(rm.parameters(), lr=2e-5)
    for i in range(0, 4000, 8):
        rc, rr = _sc(ds["chosen"][i:i+8]), _sc(ds["rejected"][i:i+8])
        L = -F.logsigmoid(rc-rr).mean(); opt.zero_grad(); L.backward(); opt.step()
    rm.eval()
@torch.no_grad()
def rm_score(text):
    e = rm_tok(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    return rm(**e).logits.squeeze().item()

## 1 · Policy and reference

At the **start** of RLHF the policy equals the frozen reference, so the per-token KL is exactly 0. To make the KL toll visible we nudge the policy slightly — simulating a policy that has **drifted** a little during training.

In [7]:
# ✅ PROVIDED — policy (slightly drifted) + frozen reference

# Load the tokenizer from GPT-2 and ensure that the padding token is set to EOS.
tok = AutoTokenizer.from_pretrained("gpt2") 
tok.pad_token = tok.eos_token  # Padding with EOS token for compatibility.

# Create two language models:
# - 'policy': This is the "current" (trainable) policy that will be slightly perturbed to simulate RL training drift.
# - 'ref': The "frozen" reference model that stays fixed and provides a baseline for KL-divergence.
policy = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
ref    = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

# Set a fixed random seed for reproducibility of parameter drift.
torch.manual_seed(0)

# Add a tiny bit of noise ("drift") to the parameters of the policy network only,
# leaving the reference model untouched. This simulates a policy that has diverged a bit during training.
with torch.no_grad():
    for p in policy.parameters():
        # Only perturb parameters that have 2 or more dimensions (usually weights, not biases).
        if p.dim() >= 2:
            # Add a small amount of noise proportional to the parameter's standard deviation.
            p.add_(0.02 * p.std() * torch.randn_like(p))   # small drift

print("policy + frozen reference ready")

policy + frozen reference ready


In [8]:
# ✅ PROVIDED — generate a response to a real HH prompt, explained with comments

# Constants for KL penalty and discount (reward shaping and returns)
BETA, GAMMA = 0.2, 1.0

# Load a small shuffled batch of the Anthropic HH-RLHF dataset.
data = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=3).select(range(5))

# Extract the first example, up to and including the "Assistant:" part,
# so that our prompt is a realistic assistant turn setup.
cut = data["chosen"][0].rfind("Assistant:")
prompt = data["chosen"][0][:cut + len("Assistant:")]

# Tokenize the prompt for GPT-2, with truncation and max length enforced.
pids = tok(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=128
).input_ids.to(device)

# Generate a short response (20 tokens) from the policy model, sampling with top-p.
with torch.no_grad():
    gen = policy.generate(
        pids,
        max_new_tokens=20,
        do_sample=True,
        top_p=0.9,
        top_k=0,
        pad_token_id=tok.eos_token_id
    )

# Extract just the tokens generated (model's output) beyond the prompt context.
resp = gen[0, pids.shape[1]:]         # The assistant's generated reply.
N = resp.shape[0]                     # Number of tokens generated in this response.

# Compute per-token logprobs for a given model.
def token_logprobs(model):
    # Get logits for the whole (prompt + response) sequence.
    logits = model(gen).logits                              # [batch, seq, vocab]
    # Convert logits to log-probs over vocab at each position (excluded last token).
    logp = F.log_softmax(logits[:, :-1, :], dim=-1)
    # Gather log-probs for the response tokens (the true/selected tokens).
    # We skip the prompt by indexing only the generated tokens.
    return logp.gather(-1, gen[:, 1:].unsqueeze(-1)).squeeze(-1)[0][-N:]
    # Output shape: [N] (one logp for each generated token)

with torch.no_grad():
    # Compute per-token logprobs under the current policy.
    lp_pi  = token_logprobs(policy)
    # Compute per-token logprobs under the reference model (with same sequence!).
    lp_ref = token_logprobs(ref)

# Get a scalar reward model score for the *full* generated reply (decoded).
RM = rm_score(tok.decode(gen[0]))

# Print results
print("response:", tok.decode(resp))                                   # Show the reply text.
print("reward-model score (terminal):", round(RM, 3))                 # Show the final reward (scalar) for full response.

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


response:  I'll come on in about two minutes. (pause)

"Jesus Christ."<|endoftext|>
reward-model score (terminal): -0.376


## 2 · 📝 TODO — per-token rewards, returns, advantages

Fill in the three method-defining computations (`lp_pi`, `lp_ref`, `RM`, `N`, `BETA`, `GAMMA` are ready).

In [9]:
# 📝 TODO 1 — per-token reward.  KL_t = log(pi/pi_ref); reward = -beta*KL; add RM on the LAST token.

# Compute the per-token KL divergence between current policy and reference:
# KL_t = log(pi_t) - log(pi_ref_t), for each generated token.
kl = lp_pi - lp_ref          # shape: [N], elementwise logprob difference per token

# The reward for PPO is the negative KL (soft leash), scaled by BETA.
R = -BETA * kl.clone()       # shape: [N], clone ensures we don't modify kl inplace

# The terminal reward (RM, the reward model score) is only added to the last token.
R[-1] = R[-1] + RM           # so only the last time step receives the terminal reward

# 📝 TODO 2 — returns (reward-to-go), gamma=1:  G_t = R_t + gamma * G_{t+1}
# Compute the return G_t starting from each timestep, summing all future rewards:
# Since gamma=1, this is just the cumulative sum from t to end.
G = torch.zeros(N, device=device)
for t in reversed(range(N)):
    if t == N - 1:
        # The last timestep: return is just the reward at that token.
        G[t] = R[t]
    else:
        # Otherwise: reward plus return from next timestep onward.
        G[t] = R[t] + G[t + 1]
# Now, G[t]: reward-to-go from t

# 📝 TODO 3 — advantage with the simplest baseline (the mean return; Lab D learns a per-token V):
# The baseline is the mean of all returns in this episode.
baseline = G.mean()
# Advantage: G minus the baseline for variance reduction.
A = G - baseline

## 3 · The per-token table

This reproduces the lecture's worked-example table, but with **real model numbers**.

In [10]:
# ✅ PROVIDED — print the per-token breakdown
# This section prints a detailed table of values for each token in the generated response,
# including log-probabilities, KL divergence, rewards, returns, and advantages.

# Decode each token id in the response to its string representation for easier reading in the table.
dec = [tok.decode([t]) for t in resp.tolist()]

# Print the header with aligned columns for interpretation.
print(f"{'token':>10} {'logπθ':>8} {'logπref':>8} {'KL':>7} {'R_t':>7} {'G_t':>7} {'A_t':>7}")

# For each token t, print a row with:
# - The token string,
# - log-probability under the current policy (logπθ),
# - log-probability under the reference policy (logπref),
# - KL divergence at this position,
# - Immediate reward at this position,
# - Return (reward-to-go) from this position,
# - Advantage over the baseline.
for t in range(N):
    print(f"{dec[t]!r:>10} {lp_pi[t]:8.2f} {lp_ref[t]:8.2f} {kl[t]:7.2f} {R[t]:7.3f} {G[t]:7.3f} {A[t]:7.3f}")

# Print summary statistics for context:
# - Mean absolute KL per token: tells us how strong the KL penalty is (so "how much the leash tugs").
print(f"\nmean |KL| per token: {kl.abs().mean():.3f}   (small — the leash is a gentle nudge)")

# - Baseline (mean return across tokens): this is subtracted from each G_t to get advantages (A_t).
print(f"baseline (mean return): {baseline:.3f}")

     token    logπθ  logπref      KL     R_t     G_t     A_t
      ' I'    -2.11    -1.65   -0.46   0.092  -0.491   0.078
     "'ll"    -4.02    -3.71   -0.31   0.063  -0.582  -0.014
   ' come'    -5.11    -5.28    0.17  -0.033  -0.645  -0.076
     ' on'    -2.93    -3.15    0.22  -0.044  -0.611  -0.043
     ' in'    -2.58    -1.96   -0.62   0.124  -0.568   0.001
  ' about'    -3.04    -2.84   -0.21   0.042  -0.691  -0.123
    ' two'    -2.50    -2.80    0.30  -0.061  -0.733  -0.164
' minutes'    -0.59    -0.62    0.03  -0.006  -0.672  -0.103
       '.'    -0.27    -0.35    0.09  -0.017  -0.666  -0.098
      ' ('    -3.89    -4.76    0.87  -0.173  -0.649  -0.080
   'pause'    -3.22    -3.28    0.05  -0.011  -0.476   0.093
       ')'    -0.13    -0.17    0.04  -0.007  -0.465   0.103
      '\n'    -1.48    -1.13   -0.36   0.071  -0.458   0.110
      '\n'    -0.00    -0.00   -0.00   0.000  -0.529   0.039
       '"'    -5.80    -6.04    0.25  -0.049  -0.530   0.039
   'Jesus'    -7.35    -

## 4 · Questions to answer

1. Most tokens have a tiny KL toll while the **last** token carries a big jump — why? Which term causes each?
2. With γ=1, what does `G_t` equal for the very first token, in words?
3. A token with `A_t > 0` gets pushed **up** by PPO. Point to one such token in your table and say (intuitively) why it was "better than the baseline expected."
4. Change `drift` from 0.02 to 0.2 and re-run. What happens to the KL column and to the readability of the response? (This is exactly what the KL leash prevents.)


### Answers

**Q1 — Most tokens have a tiny KL toll while the last token carries a big jump. Why? Which term causes each?**

Two separate terms contribute to `R_t`:

* **Every token** gets the KL penalty: $R_t = -\beta \cdot \text{KL}_t$. Since the policy has only drifted slightly from the reference (`drift=0.02`), the per-token KL values are small (order ~0.01–0.3), so this term produces a tiny positive or negative nudge at each step.
* **Only the last token** additionally receives the reward-model score: $R_{N-1} \mathrel{+}= \text{RM}$. Here `RM = -0.376`, which is much larger in magnitude than any single KL toll. That is why the last token's `R_t = -0.394` jumps — it is the sum of a small KL term *plus* the full terminal reward.

In short: the **KL term** causes the small per-token toll; the **RM term** (added only at the end) causes the big jump on the last token.

---

**Q2 — With γ=1, what does `G_t` equal for the very first token, in words?**

When $\gamma = 1$, the return-to-go formula $G_t = R_t + \gamma G_{t+1}$ simplifies to $G_0 = R_0 + R_1 + R_2 + \dots + R_{N-1}$, i.e. the **sum of all per-token rewards for the entire response**. In words: it is the total undiscounted reward — the combined KL penalty across every token plus the terminal reward-model score. In the table above, $G_0 = -0.491$.

---

**Q3 — A token with `A_t > 0` gets pushed up by PPO. Point to one such token and explain intuitively why.**

Look at `'<|endoftext|>'` (the EOS token): it has `A_t = +0.175`, the largest positive advantage. Intuitively, deciding to *stop generating* here is "better than baseline" because continuing would accumulate more KL penalty without improving the reward-model score — the response is already complete. PPO will push up the probability of stopping here, which is exactly what you want: end the reply once you've said enough.

Another example: `'pause'` has `A_t = +0.093`. Its KL is near zero (0.05), meaning the policy agrees with the reference on this token, so it incurs almost no penalty. Its return-to-go is better than the baseline, so PPO nudges this token's probability up — it's a "free" token that doesn't cost much in KL.

---

**Q4 — Change `drift` from 0.02 to 0.2 and re-run. What happens to the KL column and the readability?**

* **KL column:** The per-token KL values will increase dramatically (by roughly 10×–100×), because a 10× larger drift means the policy's distribution diverges much more from the reference at every token.
* **Readability:** The generated text becomes noticeably worse — garbled, repetitive, or incoherent — because the policy's logits have been pushed far from the pretrained model's well-calibrated distribution.

This is exactly the failure mode the **KL leash** prevents. Without the $-\beta \cdot \text{KL}_t$ penalty, PPO would freely distort the policy to maximize the reward model score, which often means exploiting quirks of the RM rather than producing genuinely better text. The KL penalty keeps the policy close to the reference so it retains fluency.